**Quickstart: direct normalisation (steps 1-2), one visit and a dataset**
What a site (e.g. a hospital) runs on its own data with `Script/neonorm.py`: for every visit **step 1**, preprocessing, and **step 2**, the direct normalisation to the BCP templates of its age and to MNI. This notebook only **runs** the pipeline: all visits go through steps 1-2 at once. Their QC: `1.quickstart_qc.ipynb` (the same `DATA_ROOT`); a run can be checked as soon as its step 2 has finished. **Step 3**, the visit from its BCP template to MNI along the chain of BCP templates (`2.full_pipeline.ipynb`), can follow later and elsewhere: it needs only the outputs of steps 1-2 and the template edges.

| step | function | parts | main output (`<DATA_ROOT>/<id>/`) |
|---|---|---|---|
| 1 | `preprocess` | 1a orientation (temporary) + rigid T2 on the T1 grid, 1b brain mask (SynthStrip, or an own mask), 1c skull stripping + N4; once per brain mask | `<prefix>_T1w_brain_mask.nii.gz`, `<prefix>_T1w_brain_n4.nii.gz`; with a T2 `<prefix>_T2w_rigid2T1.nii.gz` (+ `_0GenericAffine.mat`), `<prefix>_T2w_brain_n4.nii.gz`; `<prefix>_preprocess.json`; `<prefix>` = `<id>` (SynthStrip) or `<id>_manual_mask` (own mask) |
| 2 | `direct_normalize` | registration of the N4 T1 of step 1 to each template of `direct_templates(age)`, once per run of step 1 (the QC of steps 1-2: `1.quickstart_qc.ipynb`) | `<prefix>_direct_to-<template>_*`, `<prefix>_direct_normalize.json` |

Sections: 0 setup, 1-2 steps 1 and 2 for one visit, here in Python, 3 all visits on a Slurm cluster (the dataset table, then steps 1 and 2), 4 additional: SynthStrip only, and the comparison with a manual mask. All options: `README.md`.

# 0. Setup
**What you fill in** (the rest runs as it is): the block `FILL IN` of the cell below; the inputs of one visit in sections 1 and 2; your table in 3.1; `SUBMIT_STEP1` / `SUBMIT_STEP2 = True` in 3.2 / 3.3 to submit.

## Installation (once per computer; on the DCCN cluster done)
It writes `neonorm_env.sh`, the `ENV_FILE` of the cell below. On the server (Linux x86-64), in the folder of the repository, with FSL 6.0.6.2 and FreeSurfer 8.0.0 installed and conda on the `PATH`:
```bash
export FSLDIR=/path/to/fsl FREESURFER_HOME=/path/to/freesurfer
bash Script/setup_neonorm.sh /path/to/toolbox/neonorm user@server:/path/to/NeonatalNormalization/templates
```
- first argument: the folder for the software (made if missing): the conda environment `neonorm`, ANTs 2.6.2, the SynthStrip models, and `neonorm_env.sh`;
- second argument: a `templates/` folder with all templates (a path or `user@server:/path`), copied to `<repository>/templates` and checked.

Then run the notebooks with the kernel `<conda>/envs/neonorm/bin/python`.


In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

# ===== FILL IN (on the DCCN cluster nothing to change) =====
ENV_FILE = Path('/path/to/neonorm_env.sh')  # written by setup_neonorm.sh
EXAMPLE = Path('example')    # the folder of participants.tsv (3.1); relative to docs/tutorials
DATA_ROOT = EXAMPLE / 'out'  # the outputs: one folder per visit, the Slurm logs
N_THREADS = 8                # ANTs threads in this notebook (a Slurm job uses the CPUs it asks for)
# ============================================================

SETUP = f'source {ENV_FILE}'  # the setup line of the Slurm job scripts
assert ENV_FILE.exists(), f'ENV_FILE not found: {ENV_FILE}'
# the settings of ENV_FILE in this notebook
env = subprocess.run(['bash', '-c', f'source {ENV_FILE} && env -0'], capture_output=True, check=True).stdout
os.environ.update(kv.split('=', 1) for kv in env.decode().split('\0') if '=' in kv)
os.environ['NEONORM_NTHREADS'] = str(N_THREADS)
# work in <repository>/docs/tutorials (the kernel may start in another folder)
nb_file = globals().get('__vsc_ipynb_file__')
REPO = Path(nb_file).resolve().parents[2] if nb_file else Path(os.environ['NEONORM_BCP_TEMPLATES']).parents[1]
os.chdir(REPO / 'docs' / 'tutorials')
EXAMPLE, DATA_ROOT = EXAMPLE.resolve(), DATA_ROOT.resolve()  # absolute: the jsons record the paths
sys.path.insert(0, str(REPO / 'Script'))
import neonorm

problems = neonorm.check_environment(verbose=True)
print('\nready' if not problems else '\nnot ready:\n  - ' + '\n  - '.join(problems))


# 1. Step 1: preprocessing (`preprocess`)
`preprocess(t1, age_months, out_dir, t2=None, prefix=None, brain_mask=None)`; command line `neonorm.py --preprocess <t1> <age_months> <out_dir> [--t2 <t2>] [--prefix <prefix>] [--brain_mask <own mask>]` (`--preprocess` optional: step 1 is the default)

**Inputs**:

| argument | meaning |
|---|---|
| `t1` | path of the T1w (NIfTI) as acquired: whole head, not skull-stripped, any orientation; only read |
| `age_months` | age at scan in **months** (a number, e.g. 4.5; not weeks or years): chooses the SynthStrip models (below 12 months the infant model too). From the postmenstrual age in weeks: `neonorm.age_months_from_pma(pma_weeks)` (0 before 40 weeks) |
| `out_dir` | folder for the outputs of the visit (made if missing) |
| `t2` | optional, **not used for now** (here `None`): the T2w of the same session as acquired; with it, 1a registers it rigidly to the T1 and 1c skull-strips and bias-corrects it (for the SUITPy cerebellum of the QC, and a later T1 + T2 registration) |
| `prefix` | start of the output names (optional; default from the T1 name: `sub-001_T1w.nii.gz` -> `sub-001`); a run with an own mask needs its own, `<id>_manual_mask`, so that the SynthStrip run of the visit stays next to it |
| `brain_mask` | optional: an own brain mask instead of SynthStrip (e.g. drawn or corrected by hand), on the T1 as acquired: binary or a skull-stripped T1 (non-zero voxels = brain), same shape and affine as `t1`; on another grid it is an error (bring it to the T1 first, e.g. `antsApplyTransforms -d 3 -i <mask> -r <t1> -o <out> -n NearestNeighbor`) |

| part | function | what | output (`<out_dir>`) |
|---|---|---|---|
| 1a | `orient` | orientation: `fslreorient2std` (FSL) only if an image is not in the standard orientation (lossless, world coordinates unchanged), as temporary files for 1a-1c (not kept); with a T2: the T2 registered rigidly to the T1 (ANTsPy, whole head), on the T1 grid; a run with an own mask reuses the rigid T2 of the SynthStrip run of the visit (same T1 and T2; if there is none yet, it makes its own) | with a T2: `<prefix>_T2w_rigid2T1.nii.gz`, `_0GenericAffine.mat` |
| 1b | `synthstrip_age_specific` / `own_brain_mask` | brain mask: SynthStrip (FreeSurfer `mri_synthstrip`; below 12 months the union of the default and the infant model, from 12 months the default model), or with `brain_mask` the own mask: reoriented like the T1 if the T1 is not in the standard orientation, checked to lie on the T1, made binary | `<prefix>_T1w_brain_mask.nii.gz` |
| 1c | `head2brain_and_n4correction` | skull stripping (ANTs `ImageMath m`: image x brain mask) and bias correction (`N4BiasFieldCorrection`, inside the brain mask) of the T1 and the rigid T2 | `<prefix>_T1w_brain_n4.nii.gz`, `<prefix>_T2w_brain_n4.nii.gz` |

**Orientation**: all outputs lie on the grid of the T1 in the standard orientation; the reoriented T1 and T2 themselves are temporary (`orient.t1_reoriented` in the json says whether the T1 was reoriented). Reorienting is lossless and gives the same image every time, so a later run with an own mask, reoriented then, lies on the same grid.

**Record**: `<prefix>_preprocess.json`: inputs, software, `orient`, `skull_strip`, `head2brain` (files). **Step 2 uses** `<prefix>_T1w_brain_n4.nii.gz` (and records the mask and the rigid T2 for the QC). **Resources**: the job array of 3.2 asks for 2 CPUs and 24 GB per run (`neonorm.PREPROCESS_JOB`); less may do (not measured yet).

**Example: one visit, here in Python** (~5 min each subject). The inputs are given directly; the brain mask: SynthStrip, or an own mask (the line in comments). With a Slurm cluster, for many visits from a table: section 3 (3.2).

In [ ]:
# ===== FILL IN: the inputs of one visit =====
# here sub-ex01 of the example (example/README.md): 1.0 month, T1 + T2; the other example, sub-ex02: 8.8 months, T1 only
t1 = EXAMPLE / 'sub-ex01_T1w.nii.gz'   # as acquired (whole head, any orientation)
t2 = None                              # the T2 (e.g. EXAMPLE / 'sub-ex01_T2w.nii.gz'): an option, not used for now
age_months = 1.0
out_dir = DATA_ROOT / 'sub-ex01'
# the brain mask and the prefix of the run: SynthStrip (brain_mask None), or an own mask on the T1 as acquired (here
# the manual mask of 4.2; reoriented like the T1 if needed) with a prefix of its own
brain_mask, prefix = None, 'sub-ex01'
# brain_mask, prefix = EXAMPLE / 'sub-ex01_T1w_manual_mask.nii.gz', 'sub-ex01_manual_mask'

prepro = neonorm.preprocess(t1, age_months, out_dir, t2=t2, prefix=prefix, brain_mask=brain_mask)
pd.Series({**prepro['orient'], **{k: v for k, v in prepro['skull_strip'].items() if k != 'age_months'}, **prepro['head2brain']})


# 2. Step 2: direct normalisation (`direct_normalize`)
`direct_normalize(t1_brain, age_months, out_dir=None, prefix=None, templates=None)`; command line `neonorm.py --direct_normalize <prefix>_T1w_brain_n4.nii.gz <age_months> <out_dir> --prefix <prefix> --templates <names>`

**Inputs**:

| argument | meaning |
|---|---|
| `t1_brain` | `<prefix>_T1w_brain_n4.nii.gz` of step 1: the skull-stripped, bias-corrected T1 of one run |
| `age_months` | age at scan in **months**, as in step 1: chooses the templates |
| `out_dir` | folder of the visit, the same as in step 1 (default: the folder of `t1_brain`) |
| `prefix` | the prefix of the run of step 1: `<id>` (SynthStrip) or `<id>_manual_mask` (own mask) (default: from the name of `t1_brain`) |
| `templates` | template names; here always `direct_templates(age_months)`, the BCP templates of the visit's age window and MNI (below) |

The brain mask and the rigid T2 of the run are not needed for the registrations; `direct_normalize` records them from `<prefix>_preprocess.json` for the QC (`1.quickstart_qc.ipynb`) and step 3.

**Brain masks**: the brain mask comes from the run of step 1: SynthStrip (prefix `<id>`), or an own mask, if one is given (column `brain_mask` of the table; prefix `<id>_manual_mask`). Step 2 registers the N4 T1 of the run it is given.

| function | what | output (`<out_dir>`) |
|---|---|---|
| `register_to_template` | registration of the N4 T1 to each template: `antsRegistrationSyN.sh` default (rigid + affine + SyN), T1 only, starting from a rotation search (`antsAI`) against the nearest BCP template | per template `<prefix>_direct_to-<template>_0GenericAffine.mat`, `_1Warp.nii.gz` (visit -> template: `[warp, affine]`), `_1InverseWarp.nii.gz` (template -> visit), `_Warped.nii.gz`, `_init.mat` |

**Templates** (`direct_templates(age)`): the BCP template of the nearest atlas month (first: the start of the chains of step 3), the other templates of the visit's avgsubj edges (`avgsubj_edges(age)`: usually the two atlas months around the age, three at an atlas month; from 60 months BCP60M only), and MNI. E.g. before term (age 0): BCP00M, BCP01M, MNI; 5.55 months: BCP06M, BCP05M, BCP07M, MNI; 97 months: BCP60M, MNI. With these, every way of step 3 can be run later, and the visit can join the avgsubj averages of its age window.

**Record**: `<prefix>_direct_normalize.json` (inputs, software, the N4 images of step 1, per template the files and the `antsRegistration` calls). **Step 3 uses** this json. A template can be registered again alone (`templates`). **Resources**: 8 CPUs, 16 GB, 7-14 minutes per template; about 0.5 GB per template (the warps).

**Example: one visit, here in Python**: the N4 T1 of the run of step 1 above; about 10 min per template (4 templates at 1 month). With a Slurm cluster, for many visits or to go on meanwhile: section 3 (3.3).

In [ ]:
# ===== FILL IN: the inputs of one visit =====
# here sub-ex01 of the example (example/README.md): 1.0 month, T1 + T2; the other example, sub-ex02: 8.8 months, T1 only
age_months = 1.0
out_dir = DATA_ROOT / 'sub-ex01'
prefix = 'sub-ex01'                                   # the run of step 1: sub-ex01 (SynthStrip) or sub-ex01_manual_mask
t1_brain = out_dir / f'{prefix}_T1w_brain_n4.nii.gz'  # 1c of that run
# The templates will used
print(f'{prefix} will be warped to {neonorm.direct_templates(age_months)}')
norm = neonorm.direct_normalize(t1_brain, age_months, out_dir, prefix=prefix, templates=neonorm.direct_templates(age_months))
display(pd.DataFrame({name: {k: r[k] for k in ('affine', 'warp', 'warped')} for name, r in norm['normalize'].items()}).T)


# 3. All visits (Slurm)
The visits of a dataset from a table; each step as one Slurm job array.

## 3.1 The dataset: one row per visit
neonorm works on visits (one scan session), each in its own folder `<DATA_ROOT>/<id>/`, all files named `<id>_*`. Here the example visits, `example/participants.tsv`; for your own data write a table with the same columns:

| column | meaning |
|---|---|
| `participant_id` | the visit id and file prefix, e.g. `sub-001` (unique; a child with three visits has three ids) |
| `age_months` | age at scan in months (chooses the SynthStrip models and the BCP templates); or instead `pma_weeks`, the postmenstrual age at scan in weeks: `age_months = neonorm.age_months_from_pma(pma_weeks)`, the age after term (0 before 40 weeks: BCP00M and BCP01M) |
| `t1` | the T1w (NIfTI, whole head, not skull-stripped); only read, never changed |
| `t2` | the T2w of the same session (optional, empty if none); not used for now (3.2 does not pass it) |
| `brain_mask` | an own brain mask on the T1 as acquired (optional column, empty if none; section 1): one more run of steps 1 and 2, prefix `<id>_manual_mask` |

**Example data**: `sub-ex01`, an infant of 1.0 month (T1 and T2), and `sub-ex02`, an infant of 8.8 months (T1 only), from the open Baby Open Brains dataset (CC-BY-4.0; source and citation: `example/README.md`). Their images are already ACPC-aligned (1 mm), the T2 on the T1 grid.

In [ ]:
visits = pd.read_csv(EXAMPLE / 'participants.tsv', sep='\t', keep_default_na=False)
if 'pma_weeks' in visits:  # postmenstrual age in weeks instead of age_months
    visits['age_months'] = neonorm.age_months_from_pma(visits['pma_weeks'].astype(float))
for c in ('t1', 't2', 'brain_mask'):  # absolute paths (relative ones are relative to the table)
    if c in visits:
        visits[c] = [str(EXAMPLE / f) if f else '' for f in visits[c]]
visits['out_dir'] = [str(DATA_ROOT / i) for i in visits['participant_id']]
assert visits['participant_id'].is_unique
DATA_ROOT.mkdir(parents=True, exist_ok=True)
visits

## 3.2 Step 1, all visits
Every run of step 1 without its `<prefix>_preprocess.json` yet (`REDO_STEP1 = True`: all runs): `<id>` (SynthStrip) for every visit, and `<id>_manual_mask` for a visit with an own mask in the table; the T2 of the table is not passed for now. `submit_preprocess_array(runs, run_dir, setup)` submits them as one job array `prep` (one task per run; 2 CPUs, 24 GB each, `neonorm.PREPROCESS_JOB`), at most `max_running` tasks at once; per run the job script and its full log in `<out_dir>/log/`. `SUBMIT_STEP1 = True` submits.


In [ ]:
REDO_STEP1 = True
SUBMIT_STEP1 = False  # True: submit


def step1_runs(v):
    """The runs of step 1 of a visit: <id> (SynthStrip) and, with an own mask in the table, <id>_manual_mask (no T2
    for now)."""
    runs = [{'t1': v.t1, 'age_months': float(v.age_months), 'out_dir': v.out_dir, 'prefix': v.participant_id}]
    if getattr(v, 'brain_mask', ''):
        runs.append({**runs[0], 'prefix': f'{v.participant_id}_manual_mask', 'brain_mask': v.brain_mask})
    return runs


runs = [r for v in visits.itertuples() for r in step1_runs(v)
        if REDO_STEP1 or not (Path(r['out_dir']) / f"{r['prefix']}_preprocess.json").exists()]
print(f'{len(runs)} runs of step 1 to do:', [r['prefix'] for r in runs])
if SUBMIT_STEP1 and runs:
    arr = neonorm.submit_preprocess_array(runs, DATA_ROOT / 'logs' / datetime.now().strftime('step1_%Y%m%d_%H%M%S'),
                                          setup=SETUP, max_running=50)
    print('array', arr['job_id'])


## 3.3 Step 2, all visits
Every run of step 1 with its N4 T1 (`<id>`, and `<id>_manual_mask` for a visit with an own mask; once the arrays of 3.2 have finished); runs with all their templates registered already are left out (`REDO_STEP2 = True`: all runs). `submit_direct_normalize_array(...)` submits one job array `direct` (one task per run), at most `max_running` tasks at once; per run the job script and full log in `<out_dir>/log/`.

In [ ]:
SUBMIT_STEP2 = True  # True: submit
REDO_STEP2 = False


def registered(a):
    """True if every template of the run is in its <prefix>_direct_normalize.json."""
    f = Path(a['out_dir']) / f"{a['prefix']}_direct_normalize.json"
    return f.exists() and set(a['templates']) <= set(json.loads(f.read_text()).get('normalize', {}))


def step2_runs(v):
    """The direct_normalize runs of a visit, one per run of step 1: <id> (SynthStrip) and, with an own mask in the
    table, <id>_manual_mask; each to direct_templates(age)."""
    prefixes = [v.participant_id] + ([f'{v.participant_id}_manual_mask'] if getattr(v, 'brain_mask', '') else [])
    return [{'t1_brain': str(Path(v.out_dir) / f'{p}_T1w_brain_n4.nii.gz'), 'age_months': float(v.age_months),
             'out_dir': v.out_dir, 'prefix': p, 'templates': neonorm.direct_templates(v.age_months)} for p in prefixes]

# Check whether step 1 finished
runs = [a for v in visits.itertuples() for a in step2_runs(v)]
print('runs of step 1 not done or failed (no N4 T1):', [a['prefix'] for a in runs if not Path(a['t1_brain']).exists()])
# Run step 2
runs = [a for a in runs if Path(a['t1_brain']).exists() and (REDO_STEP2 or not registered(a))]
print(f'{len(runs)} runs of step 2')
if SUBMIT_STEP2 and runs:
    arr = neonorm.submit_direct_normalize_array(runs, DATA_ROOT / 'logs' / datetime.now().strftime('step2_%Y%m%d_%H%M%S'),
                                                setup=SETUP, max_running=50)
    print('array', arr['job_id'])

# 4. Additional: SynthStrip only, and the comparison with a manual mask
Apart from the pipeline. For only the SynthStrip brain mask, step 1 can stop at its part 1b: `synthstrip_age_specific(t1, age_months, out_dir, prefix)` on the T1 as acquired gives `<prefix>_T1w_brain_mask.nii.gz` on the grid of that T1 (no reorientation, no N4). The prefix `<id>_synthstrip_only` keeps it apart from the files of step 1. A manual mask drawn on the same T1 lies on the same grid, so the two masks are compared directly.

## 4.1 One visit


In [ ]:
# ===== FILL IN: the inputs of one visit =====
t1 = EXAMPLE / 'sub-ex01_T1w.nii.gz'   # as acquired (whole head, any orientation)
age_months = 1.0
out_dir = DATA_ROOT / 'sub-ex01'
prefix = 'sub-ex01_synthstrip_only'

neonorm.synthstrip_age_specific(t1, age_months, out_dir, prefix=prefix)


## 4.2 The SynthStrip mask against the manual mask
The visit of 4.1 and its manual mask, drawn on the same T1 as acquired: Dice of the two masks, the HD95 of their surfaces (mm, and log10 as in the QC), their volumes, and the voxels only in one of them (% of the SynthStrip volume). Here the manual mask of sub-ex01 is the manually corrected segmentation of BOBs, all labels (example/README.md): brain tissue and ventricles, without the CSF around the brain, so SynthStrip is expected to be somewhat larger.


In [ ]:
import nibabel as nib
import numpy as np

manual = EXAMPLE / 'sub-ex01_T1w_manual_mask.nii.gz'  # FILL IN: the manual mask of the visit of 4.1
synth = out_dir / f'{prefix}_T1w_brain_mask.nii.gz'   # 4.1
s_img, m_img = nib.load(synth), nib.load(manual)
assert s_img.shape[:3] == m_img.shape[:3] and np.allclose(s_img.affine, m_img.affine, atol=1e-3), \
    'the manual mask is not on the grid of the T1'
s, m = (np.squeeze(np.asanyarray(i.dataobj)) > 0 for i in (s_img, m_img))
zooms = s_img.header.get_zooms()[:3]
vox = float(np.prod(zooms)) / 1000  # ml per voxel
hd95 = neonorm._hd95(s, m, zooms)    # mm, as in the QC of the registrations
mask_eval = pd.Series({'dice': 2 * (m & s).sum() / (m.sum() + s.sum()), 'hd95_mm': hd95, 'log10_hd95': np.log10(hd95),
                       'synthstrip_ml': s.sum() * vox, 'manual_ml': m.sum() * vox,
                       'only_synthstrip_pct': 100 * (s & ~m).sum() / s.sum(), 'only_manual_pct': 100 * (m & ~s).sum() / s.sum()})
mask_eval.round(3)


## 4.3 Figure (an example)
The two masks of the visit in the style of Figure 1 of the project (sans-serif 5-7 pt, 183 mm wide): the T1 through the brain centre in three views, in blue the voxels only in the SynthStrip mask, in red those only in the manual mask, the outline of the manual mask in black; above, the values of 4.2. The images are shown in RAS (`nib.as_closest_canonical`), whatever the orientation of the T1 as acquired.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch

canon = lambda f: nib.as_closest_canonical(nib.load(f))  # RAS: the array axes are left-right, back-front, down-up
t1_img = canon(t1)
gray = np.squeeze(np.asanyarray(t1_img.dataobj)).astype(np.float32)
s_c, m_c = (np.squeeze(np.asanyarray(canon(f).dataobj)) > 0 for f in (synth, manual))
brain = s_c | m_c
gray = np.clip(gray / np.percentile(gray[brain], 99.5), 0, 1)
c = [int(round(x)) for x in np.argwhere(brain).mean(0)]  # brain centre
z = t1_img.header.get_zooms()[:3]
views = {'Sagittal': (lambda a: a[c[0]].T, z[2] / z[1]), 'Coronal': (lambda a: a[:, c[1]].T, z[2] / z[0]),
         'Axial': (lambda a: a[:, :, c[2]].T, z[1] / z[0])}  # (slice, aspect of its voxels)
ONLY = {'only SynthStrip': (s_c & ~m_c, neonorm.WAY_COLORS['tpl2tpl']), 'only manual': (m_c & ~s_c, neonorm.WAY_COLORS['avgsubj'])}
MM = 1 / 25.4
rc = {'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
      'pdf.fonttype': 42, 'font.size': 6}
with plt.rc_context(rc):
    fig, axes = plt.subplots(1, len(views), figsize=(183 * MM, 72 * MM))
    for ax, (view, (cut, aspect)) in zip(axes, views.items()):
        ys, xs = np.nonzero(cut(brain))
        box = (slice(ys.min() - 3, ys.max() + 4), slice(xs.min() - 3, xs.max() + 4))  # around the brain
        rgb = np.repeat(cut(gray)[box][..., None], 3, -1)
        for only, color in ONLY.values():
            rgb[cut(only)[box]] = to_rgb(color)
        ax.imshow(rgb, origin='lower', interpolation='nearest', aspect=aspect)
        ax.contour(cut(m_c)[box].astype(float), [0.5], colors='black', linewidths=0.5)
        ax.set_title(view, fontsize=7, pad=4)
        ax.axis('off')
    fig.suptitle(f"{prefix} vs manual mask: Dice {mask_eval['dice']:.3f}, HD95 {mask_eval['hd95_mm']:.1f} mm, "
                 f"only SynthStrip {mask_eval['only_synthstrip_pct']:.1f}%, only manual {mask_eval['only_manual_pct']:.1f}%",
                 fontsize=7)
    fig.legend([Patch(color=c) for _, c in ONLY.values()] + [plt.Line2D([], [], color='black', lw=0.5)],
               list(ONLY) + ['manual mask'], loc='lower center', ncol=3, frameon=False, fontsize=6)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    plt.show()
